# Mocap 01 -- BVH & Kinematics: Skeleton Animation from Scratch

> **Geo-Instructor . Motion Capture Career Track . Notebook 1 of 3**

Every character in a game or film has a skeleton. Every skeleton is driven by joint angles.
This notebook builds FK, quaternion slerp, and FABRIK IK entirely from scratch.

```
Runtime: ~3 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace'})
print('Ready.')

---
## Part 1 -- Skeleton as a Tree

A BVH skeleton is a hierarchy of joints. Each joint has:
- A **parent** (root has none)
- A local **offset** from parent (the bone vector at rest)
- A **rotation** applied at each frame

```
Hips
  Spine -> Chest -> Neck -> Head
         -> LeftShoulder -> LeftElbow -> LeftWrist
         -> RightShoulder -> RightElbow -> RightWrist
  LeftHip -> LeftKnee -> LeftAnkle
  RightHip -> RightKnee -> RightAnkle
```

In [ ]:
# Define a 15-joint biped skeleton as (name, parent_idx, local_offset_xyz)
JOINTS = [
 #  idx  name              parent  offset (x,y,z)
    (0,  'Hips',           -1,     [0,   0,    0   ]),
    (1,  'Spine',           0,     [0,   0.15, 0   ]),
    (2,  'Chest',           1,     [0,   0.15, 0   ]),
    (3,  'Neck',            2,     [0,   0.12, 0   ]),
    (4,  'Head',            3,     [0,   0.12, 0   ]),
    (5,  'LeftShoulder',    2,     [-0.12, 0.05, 0 ]),
    (6,  'LeftElbow',       5,     [-0.22, 0,   0  ]),
    (7,  'LeftWrist',       6,     [-0.18, 0,   0  ]),
    (8,  'RightShoulder',   2,     [ 0.12, 0.05, 0 ]),
    (9,  'RightElbow',      8,     [ 0.22, 0,   0  ]),
    (10, 'RightWrist',      9,     [ 0.18, 0,   0  ]),
    (11, 'LeftHip',         0,     [-0.10,-0.05, 0 ]),
    (12, 'LeftKnee',       11,     [0,   -0.25, 0  ]),
    (13, 'LeftAnkle',      12,     [0,   -0.25, 0  ]),
    (14, 'RightHip',        0,     [ 0.10,-0.05, 0 ]),
    (15, 'RightKnee',      14,     [0,   -0.25, 0  ]),
    (16, 'RightAnkle',     15,     [0,   -0.25, 0  ]),
]
offsets = np.array([j[3] for j in JOINTS], dtype=float)
parents = [j[2] for j in JOINTS]
names   = [j[1] for j in JOINTS]
print(f'{len(JOINTS)} joints defined')

---
## Part 2 -- Forward Kinematics

FK: given local joint rotations, compute world positions.
```
  world_transform[joint] = world_transform[parent] @ local_rotation @ translation
```
We use 4x4 homogeneous matrices throughout.

In [ ]:
def rot_x(a): c,s=np.cos(a),np.sin(a); return np.array([[1,0,0,0],[0,c,-s,0],[0,s,c,0],[0,0,0,1]])
def rot_y(a): c,s=np.cos(a),np.sin(a); return np.array([[c,0,s,0],[0,1,0,0],[-s,0,c,0],[0,0,0,1]])
def rot_z(a): c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0,0],[s,c,0,0],[0,0,1,0],[0,0,0,1]])
def translate(v): T=np.eye(4); T[:3,3]=v; return T

def forward_kinematics(offsets, parents, joint_rotations):
    """Compute world positions for all joints.
    joint_rotations: (N, 4x4) local rotation matrices
    Returns: world_pos (N, 3)"""
    n = len(offsets)
    world_transforms = [np.eye(4)] * n
    for i, (off, par) in enumerate(zip(offsets, parents)):
        local = translate(off) @ joint_rotations[i]
        if par < 0:
            world_transforms[i] = local
        else:
            world_transforms[i] = world_transforms[par] @ local
    world_pos = np.array([T[:3, 3] for T in world_transforms])
    return world_pos, world_transforms

# T-pose (all rotations = identity)
t_pose_rots = [np.eye(4)] * len(JOINTS)
pos_t, _ = forward_kinematics(offsets, parents, t_pose_rots)

# Walking pose: slight hip rotation, bent knees, swinging arms
walking_rots = [np.eye(4)] * len(JOINTS)
walking_rots[0]  = rot_y(0.05)                          # hips sway
walking_rots[11] = rot_x(np.radians(20))                # left hip forward
walking_rots[12] = rot_x(np.radians(-35))               # left knee bend
walking_rots[14] = rot_x(np.radians(-15))               # right hip back
walking_rots[6]  = rot_x(np.radians(-30))               # left elbow
walking_rots[9]  = rot_x(np.radians(30))                # right elbow
pos_walk, _ = forward_kinematics(offsets, parents, walking_rots)

def draw_skeleton(ax, pos, parents, title, color='steelblue'):
    for i, par in enumerate(parents):
        if par >= 0:
            ax.plot([pos[par,0],pos[i,0]], [pos[par,2],pos[i,2]], [pos[par,1],pos[i,1]],
                    '-', color=color, lw=2)
    ax.scatter(pos[:,0], pos[:,2], pos[:,1], s=20, c=color, zorder=5)
    ax.set_title(title, fontsize=9)

fig = plt.figure(figsize=(12, 6))
for i, (pos, title) in enumerate([(pos_t, 'T-Pose'), (pos_walk, 'Walking Pose')]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    draw_skeleton(ax, pos, parents, title)
    ax.set_xlim(-0.6,0.6); ax.set_ylim(-0.4,0.4); ax.set_zlim(-0.6,0.8)
    ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('Y')
plt.tight_layout(); plt.show()

In [ ]:
# Animate 4 keyframes of a walk cycle
n_frames = 4
fig = plt.figure(figsize=(16, 4))
fig.suptitle('Walk cycle -- 4 keyframes (FK from joint angles)', fontsize=11)
for f in range(n_frames):
    t = f / n_frames * 2 * np.pi
    rots = [np.eye(4)] * len(JOINTS)
    rots[0]  = rot_y(np.sin(t) * 0.08)
    rots[11] = rot_x(np.radians(25 * np.sin(t)))
    rots[12] = rot_x(np.radians(max(0, -40 * np.sin(t))))
    rots[13] = rot_x(np.radians(max(0, -20 * np.sin(t))))
    rots[14] = rot_x(np.radians(-25 * np.sin(t)))
    rots[15] = rot_x(np.radians(max(0, 40 * np.sin(t))))
    rots[5]  = rot_x(np.radians(-20 * np.sin(t)))  # left arm swing
    rots[8]  = rot_x(np.radians(20 * np.sin(t)))   # right arm swing
    pos, _ = forward_kinematics(offsets, parents, rots)
    ax = fig.add_subplot(1, n_frames, f+1, projection='3d')
    draw_skeleton(ax, pos, parents, f'Frame {f+1}')
    ax.set_xlim(-0.6,0.6); ax.set_ylim(-0.4,0.4); ax.set_zlim(-0.6,0.8)
plt.tight_layout(); plt.show()
print('FK is a simple tree traversal: parent transform * local rotation * local offset')

---
## Part 3 -- Quaternions & Gimbal Lock

Euler angles have **gimbal lock**: when two axes align, a degree of freedom is lost.
Quaternions (w, x, y, z) represent rotations without singularities.

**SLERP** (spherical linear interpolation) gives smooth interpolation between two rotations.

In [ ]:
def euler_to_quat(rx, ry, rz):
    """ZYX Euler -> quaternion."""
    cx,sx = np.cos(rx/2), np.sin(rx/2)
    cy,sy = np.cos(ry/2), np.sin(ry/2)
    cz,sz = np.cos(rz/2), np.sin(rz/2)
    return np.array([
        cx*cy*cz + sx*sy*sz,
        sx*cy*cz - cx*sy*sz,
        cx*sy*cz + sx*cy*sz,
        cx*cy*sz - sx*sy*cz
    ])

def quat_to_matrix(q):
    w,x,y,z = q / np.linalg.norm(q)
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-w*z),   2*(x*z+w*y)],
        [2*(x*y+w*z),   1-2*(x*x+z*z), 2*(y*z-w*x)],
        [2*(x*z-w*y),   2*(y*z+w*x),   1-2*(x*x+y*y)]
    ])

def slerp(q0, q1, t):
    q0 = q0 / np.linalg.norm(q0); q1 = q1 / np.linalg.norm(q1)
    dot = np.clip(np.dot(q0, q1), -1, 1)
    if dot < 0: q1 = -q1; dot = -dot
    if dot > 0.9995: return q0 + t*(q1-q0)
    theta0 = np.arccos(dot)
    theta = theta0 * t
    q2 = q1 - q0*dot; q2 /= np.linalg.norm(q2)
    return q0*np.cos(theta) + q2*np.sin(theta)

# Demo: rotating forearm from rest to bent, with Euler vs Slerp
q_start = euler_to_quat(0, 0, 0)
q_end   = euler_to_quat(np.radians(90), np.radians(30), 0)
ts = np.linspace(0, 1, 60)

euler_lerp = [np.array([0,0,0]) + t*np.array([np.radians(90), np.radians(30), 0]) for t in ts]
quat_slerp = [slerp(q_start, q_end, t) for t in ts]

# Extract Y euler angle from both
def quat_to_euler_y(q): return np.arcsin(2*(q[0]*q[2] - q[3]*q[1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(ts, [e[1]*180/np.pi for e in euler_lerp], 'tomato', lw=2, label='Linear Euler lerp')
ax1.plot(ts, [quat_to_euler_y(q)*180/np.pi for q in quat_slerp], 'steelblue', lw=2, label='Quaternion slerp')
ax1.set_title('Y rotation angle over interpolation'); ax1.legend(); ax1.set_xlabel('t')
# Angular speed (should be constant for slerp)
ax2.plot(ts[1:], [np.degrees(np.arccos(np.clip(np.dot(quat_slerp[i]/np.linalg.norm(quat_slerp[i]), quat_slerp[i-1]/np.linalg.norm(quat_slerp[i-1])), -1, 1))) for i in range(1, len(quat_slerp))], 'steelblue', lw=2, label='Slerp angular speed')
ax2.set_title('Angular speed per step (slerp = constant)'); ax2.legend(); ax2.set_xlabel('t')
plt.tight_layout(); plt.show()
print('Slerp: constant angular speed. Linear Euler lerp: speed varies, unnatural motion.')

---
## Part 4 -- FABRIK Inverse Kinematics

FABRIK (Forward and Backward Reaching IK) solves for joint angles given a target end-effector position.

```
Forward pass:  place end at target, reach back along chain
Backward pass: pin root, reach forward along chain
Repeat until converged
```
Simple, fast, no Jacobian needed.

In [ ]:
def fabrik(joints, lengths, target, n_iters=20, tol=1e-4):
    """FABRIK IK. joints: (n,2) initial positions. lengths: (n-1,) bone lengths."""
    joints = joints.copy()
    root = joints[0].copy()
    for it in range(n_iters):
        # Forward: end effector -> root
        joints[-1] = target.copy()
        for i in range(len(joints)-2, -1, -1):
            d = joints[i] - joints[i+1]
            d_norm = np.linalg.norm(d)
            if d_norm > 1e-8: joints[i] = joints[i+1] + d/d_norm * lengths[i]
        # Backward: root -> end effector
        joints[0] = root.copy()
        for i in range(len(joints)-1):
            d = joints[i+1] - joints[i]
            d_norm = np.linalg.norm(d)
            if d_norm > 1e-8: joints[i+1] = joints[i] + d/d_norm * lengths[i]
        if np.linalg.norm(joints[-1] - target) < tol: break
    return joints, it+1

# 3-joint arm (shoulder -> elbow -> wrist -> fingertip)
initial = np.array([[0.,0.],[0.3,0.],[0.6,0.],[0.9,0.]])
lengths = np.array([0.3, 0.3, 0.3])
targets = [
    np.array([0.7, 0.5]),
    np.array([0.2, 0.8]),
    np.array([-0.3, 0.4]),
    np.array([0.8, -0.3]),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
fig.suptitle('FABRIK IK -- 3-joint arm reaching 4 targets', fontsize=10)
for ax, target in zip(axes, targets):
    final, iters = fabrik(initial, lengths, target)
    ax.plot(initial[:,0], initial[:,1], 'o--', color='gray', lw=1, ms=6, alpha=0.5, label='Rest')
    ax.plot(final[:,0], final[:,1], 'o-', color='steelblue', lw=2.5, ms=8, label='IK result')
    ax.plot(*target, '*', color='tomato', ms=14, label='Target')
    ax.set_xlim(-0.8, 1.2); ax.set_ylim(-0.8, 1.2); ax.set_aspect('equal')
    ax.set_title(f'iters={iters}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print('FABRIK converges in very few iterations and handles long chains efficiently.')

---
## Exercises

### Exercise 1 -- Parse a real BVH file
BVH has HIERARCHY (joint tree + offsets) and MOTION (per-frame Euler angles) sections.
Parse it and run FK to animate the skeleton.

### Exercise 2 -- Jacobian IK
Compute the Jacobian J (how end-effector position changes with joint angles).
Solve delta_theta = J_pseudo_inverse * delta_pos using SVD.

### Exercise 3 -- Skinning
Given bone transforms and per-vertex bone weights (linear blend skinning),
compute deformed mesh vertex positions.

---
## Summary

| Concept | What it does | Production tool |
|---------|-------------|----------------|
| FK | joint angles -> world positions | All animation software |
| Quaternion | rotation without gimbal lock | Every game engine |
| Slerp | smooth rotation interpolation | Unity, Unreal, Blender |
| FABRIK IK | target pos -> joint angles | Blender, Unity IK |
| BVH | motion capture file format | Vicon, OptiTrack, Mixamo |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  FK  -> tree traversal with 4x4 homogeneous matrices')
print('  Quaternions -> no gimbal lock, compact, slerp-friendly')
print('  FABRIK IK   -> no Jacobian, fast, handles long chains')
print()
print('Production: Vicon/OptiTrack for capture, MotionBuilder for cleanup, Unity/Unreal for playback')